<a href="https://colab.research.google.com/github/russvanfossen/Data_Science_Portfolio/blob/main/NYC_rat_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sodapy

In [2]:
import pandas as pd
from sodapy import Socrata
from google.colab import userdata
from datetime import datetime, date

In [3]:
token = userdata.get('NYCDATA_TOKEN')

dataset_id = "p937-wjvj"
batch_size = 50_000
offset = 0
columns = None
d = '2026-10-1'
date = pd.to_datetime(d).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]

with Socrata("data.cityofnewyork.us", token, timeout=120) as client:
    # Get a consistent column order, including fields absent from some batches.
    metadata = client.get_metadata(dataset_id)
    columns = [
        col["fieldName"]
        for col in metadata["columns"]
        if not col["fieldName"].startswith(":")
    ]

    while True:
        rows = client.get(
            dataset_id,
            where=f"inspection_date > '{date}' and result like 'Fail%'",
            limit=batch_size,
            offset=offset,
            order=":id"
          )

        if not rows:
            break

        batch_df = pd.DataFrame.from_records(rows).reindex(columns=columns)

        batch_df.to_csv(
            "socrata_data.csv",
            mode="w" if offset == 0 else "a",
            header=(offset == 0),
            index=False,
        )

        offset += len(rows)
        print(f"Downloaded {offset:,} rows")

Downloaded 3,731 rows


In [4]:
df = pd.read_csv('socrata_data.csv')
df.head()

,job_id,inspection_date,inspection_type,house_number,street_name,zip_code,borough,result,letter_type,observations,...,x_coord,y_coord,latitude,longitude,location,community_board,council_district,census_tract,bin,nta
0,PC8694928,2026-10-01T12:05:00.000,Compliance,368,PUTNAM AVENUE,11216.0,Brooklyn,Failed for Rat Activity,Summons Issued,Burrows,...,NaN,NaN,40.684442,-73.946356,"{'latitude': '40.684441509568', 'longitude': '...",3.0,36.0,267.0,3051981.0,Bedford-Stuyvesant (West)
1,PC8706523,2026-10-01T13:57:35.000,Compliance,147,MARCUS GARVEY BLVD,11206.0,Brooklyn,Failed for Rat Activity,Summons Issued,"Droppings, Burrows, Runways",...,NaN,NaN,40.693521,-73.940128,"{'latitude': '40.693520884051', 'longitude': '...",3.0,36.0,281.0,3043065.0,Bedford-Stuyvesant (East)
2,PC8378096,2026-10-01T11:05:00.000,Compliance,894,RIVERSIDE DRIVE,10032.0,Manhattan,Failed for Rat Activity and Other Reason,Summons Issued,"Droppings, Burrows, Harborage",...,NaN,NaN,40.837775,-73.946857,"{'latitude': '40.837774899415', 'longitude': '...",12.0,7.0,247.0,1063327.0,Washington Heights (South)
3,PC8727871,2026-10-01T13:45:52.000,Compliance,197,8 AVENUE,10011.0,Manhattan,Failed for Other Reason,Summons Issued,Harborage,...,NaN,NaN,40.743508,-73.999794,"{'latitude': '40.743507760111', 'longitude': '...",4.0,3.0,89.0,1013263.0,Chelsea-Hudson Yards
4,PC8598017,2026-10-01T11:43:00.000,Compliance,620,EAST 13 STREET,10009.0,Manhattan,Failed for Rat Activity and Other Reason,Summons Issued,"Burrows, Rats, Animal Nuisance",...,990385.0,204674.0,40.728451,-73.977865,"{'latitude': '40.728450593495', 'longitude': '...",3.0,2.0,28.0,1004927.0,East Village


In [5]:
!pip install folium
import folium

In [6]:
nyc_coords = [40.742122160341616, -73.90645332300794]

In [7]:
map = folium.Map(location=nyc_coords, zoom_start=10, tiles="OpenStreetMap.HOT")
map

In [8]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon
marker_cluster = MarkerCluster()

In [9]:
df_dropped = df.dropna(subset=['latitude', 'longitude'])
df_dropped.head()
df_dropped.info()
df_dropped['latitude'].astype(float)
df_dropped['longitude'].astype(float)

<class 'pandas.core.frame.DataFrame'>
Index: 3606 entries, 0 to 3730
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   job_id            3606 non-null   object 
 1   inspection_date   3606 non-null   object 
 2   inspection_type   3606 non-null   object 
 3   house_number      3606 non-null   object 
 4   street_name       3606 non-null   object 
 5   zip_code          3570 non-null   float64
 6   borough           3606 non-null   object 
 7   result            3606 non-null   object 
 8   letter_type       3033 non-null   object 
 9   observations      3606 non-null   object 
 10  bbl               3556 non-null   float64
 11  boro_code         3606 non-null   int64  
 12  block             3606 non-null   int64  
 13  lot               3606 non-null   int64  
 14  x_coord           1299 non-null   float64
 15  y_coord           1299 non-null   float64
 16  latitude          3606 non-null   float64
 17  

,longitude
0,-73.946356
1,-73.940128
2,-73.946857
3,-73.999794
4,-73.977865
...,...
3725,-73.934085
3726,-73.926688
3727,-73.942697
3729,-73.941069


In [10]:
for index, record in df_dropped.iterrows():
    # Combine the result and inspection date into a readable HTML string
    popup_text = f"{record['house_number']} {record['street_name']}<br>Result: {record['result']}<br>Date: {record['inspection_date']}"
    popup = folium.Popup(popup_text, max_width=300)

    marker = folium.Marker(
        location=[record['latitude'], record['longitude']],
        icon=folium.DivIcon(html='<div style="font-size: 28px;">🐀</div>',
                            icon_size=(32, 32),
                            icon_anchor=(16, 16)),
        popup=popup
    )
    marker_cluster.add_child(marker)

In [11]:
map.add_child(marker_cluster)
map

In [12]:
map.save('map.html')